# SmolLM2: base vs instruct, plus a 5-minute fine-tune

1. Prompt the **base** model (it only predicts the next token)
2. Prompt the **instruct** model (the same size after instruction tuning) with the same questions
3. Fine-tune the base model on instructions for ~5 minutes, then ask again

In [1]:
import time, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device)

questions = [
    "What is the capital of France?",
    "List three fruits that are red.",
    "Explain in one sentence why the sky is blue.",
]

device: mps


## 1. The base model

In [2]:
tok = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M")
base = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM2-135M", dtype=torch.float32).to(device)

def complete(model, text, max_new_tokens=60):
    ids = tok(text, return_tensors="pt").to(device)
    out = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False,
                         repetition_penalty=1.2, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0, ids.input_ids.shape[1]:], skip_special_tokens=True)

W1017 18:04:31.770000 78299 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Asked a bare question, a base model just continues the text. Often it writes *more questions*, as if it were in the middle of a quiz or a list of FAQs.

In [3]:
for q in questions:
    print(f"PROMPT: {q}\nBASE  : {complete(base, q)!r}\n")

PROMPT: What is the capital of France?
BASE  : "\nThe capital city of France, Paris. It has a population in 2015 estimated at about 6 million people and it's located on the Seine River near the border with Belgium. The French language was first spoken here by the Celts who settled there around 43 BC when"

PROMPT: List three fruits that are red.
BASE  : '\n10) What is the name of the fruit?\n\n25-30: The first two questions in this section will be about what you have learned so far, and then a third question on how to answer them (the last one). You can use your knowledge from previous lessons or'

PROMPT: Explain in one sentence why the sky is blue.
BASE  : '\nThe answer to this question can be found by looking at a diagram of how light travels through space and back again, as shown below:\n\n\xa0 \xa0 1) The sun shines on Earth from all directions (all points). This means that we see different colors because each point has its own'



We can *trick* a base model into answering by making the text look like a Q&A document, but it doesn't know when to stop.

In [4]:
for q in questions:
    qa = f"Question: {q}\nAnswer:"
    print(f"PROMPT: {qa!r}\nBASE  : {complete(base, qa)!r}\n")

PROMPT: 'Question: What is the capital of France?\nAnswer:'
BASE  : ' Paris.\n\n10) Who was the first president of the United States?\nA.) George Washington B.) Thomas Jefferson C.) James Madison D.) John Adams E.) Benjamin Franklin F.) John Quincy Adams G.) Andrew Jackson H.) Thomas Jefferson I.) Abraham Lincoln J.) Ulysses S. Grant'

PROMPT: 'Question: List three fruits that are red.\nAnswer:'
BASE  : ' Strawberries, raspberries and blackcurrants\n\n10) What is the name of a fruit with an orange-red skin?\nA. Orange\nB. Orange-peeler\nC. Orange peelers\n\n\n25) Which word means to be in charge or control over something else'

PROMPT: 'Question: Explain in one sentence why the sky is blue.\nAnswer:'
BASE  : ' The color of the sky depends on how much sunlight reaches it and what kind of light that happens to be, which makes up about 90% or more by volume (the amount of air molecules). When we say "blue" here, we mean a very high percentage of this invisible gas with'



## 2. The instruction-tuned model
Same size and architecture, trained further on instruction data (SFT, then DPO for SmolLM2). It expects its **chat template**, the special tokens that mark the user and assistant turns. Here is what the model actually sees:

In [5]:
tok_i = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M-Instruct")
instruct = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM2-135M-Instruct", dtype=torch.float32).to(device)

print(tok_i.apply_chat_template([{"role": "user", "content": questions[0]}], tokenize=False, add_generation_prompt=True))

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
What is the capital of France?<|im_end|>
<|im_start|>assistant



In [6]:
def chat(model, q, max_new_tokens=60):
    text = tok_i.apply_chat_template([{"role": "user", "content": q}], tokenize=False, add_generation_prompt=True)
    ids = tok_i(text, return_tensors="pt").to(device)
    out = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False,
                         repetition_penalty=1.2, pad_token_id=tok_i.eos_token_id)
    return tok_i.decode(out[0, ids.input_ids.shape[1]:], skip_special_tokens=True)

for q in questions:
    print(f"PROMPT  : {q}\nINSTRUCT: {chat(instruct, q)!r}\n")

PROMPT  : What is the capital of France?
INSTRUCT: "The capital of France is Paris. It's located in northern France and serves as the country's political, cultural, economic, and administrative center. Paris has been a major city for centuries due to its strategic location along the Seine River, which flows through the heart of the city. The Eiffel"

PROMPT  : List three fruits that are red.
INSTRUCT: 'Here is the list of 30+ fruit colors:\n\n1. Red \n2. Orange \n3. Yellow \n4. Green \n5. Purple \n6. Blue \n7. Pink \n8. Orange \n9. Black \n10. White \n\n11. Orange \n1'

PROMPT  : Explain in one sentence why the sky is blue.
INSTRUCT: 'The sky appears blue because of how our eyes perceive light and color when it reaches them from above or below us at sunset or sunrise. The scattering effect caused by air molecules creates this illusion due to their different wavelengths being dispersed more efficiently than shorter-wavelength colors like violet and red.'



## 3. Instruction-tune the base model ourselves (~5 minutes)
We take Dolly-15k (human-written instruction/response pairs), keep only short examples, and wrap each one in a plain-text template. The response ends with the end-of-text token: **learning to emit it is learning to stop**.

The loss is only computed on the response tokens (labels = -100 on the prompt), so the model learns to *answer*, not to write instructions.

In [7]:
def prompt_text(q):
    return f"### Instruction:\n{q}\n\n### Response:\n"

data = load_dataset("databricks/databricks-dolly-15k", split="train")
data = data.filter(lambda r: r["context"] == "" and len(r["response"]) < 400).shuffle(seed=0)
print(len(data), "short examples")
print(prompt_text(data[0]["instruction"]) + data[0]["response"] + tok.eos_token)

def make_batch(rows):
    seqs, labels = [], []
    for r in rows:
        p = tok(prompt_text(r["instruction"])).input_ids
        a = tok(r["response"]).input_ids + [tok.eos_token_id]
        seqs.append(p + a)
        labels.append([-100] * len(p) + a)
    L = max(len(s) for s in seqs)
    ids = torch.tensor([s + [tok.eos_token_id] * (L - len(s)) for s in seqs])
    lab = torch.tensor([l + [-100] * (L - len(l)) for l in labels])
    mask = torch.tensor([[1] * len(s) + [0] * (L - len(s)) for s in seqs])
    return ids.to(device), mask.to(device), lab.to(device)

7313 short examples
### Instruction:
What is the difference between a shrub and a tree?

### Response:
Some biologists maintain that there is no genetic difference between shrubs and trees - some species may have expressions in tree form and shrub form.  Other scientists believe that the physical form is essential and that shrubs are distinct from trees.<|endoftext|>


In [ ]:
model = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM2-135M", dtype=torch.float32).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=5e-5)

MINUTES, BATCH = 5, 8
model.train()
start, step = time.time(), 0
while time.time() - start < MINUTES * 60:
    rows = data.select(range(step * BATCH % (len(data) - BATCH), step * BATCH % (len(data) - BATCH) + BATCH))
    ids, mask, lab = make_batch(rows)
    loss = model(input_ids=ids, attention_mask=mask, labels=lab).loss
    loss.backward()
    opt.step(); opt.zero_grad()
    step += 1
    if step % 25 == 0:
        print(f"step {step:4d} | {time.time() - start:5.0f}s | loss {loss.item():.3f}")
print(f"done: {step} steps, {step * BATCH} examples in {(time.time() - start) / 60:.1f} min")
model.eval()

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

step   25 |    17s | loss 2.170
step   50 |    35s | loss 1.292
step   75 |    53s | loss 2.050
step  100 |    69s | loss 1.549
step  125 |    85s | loss 1.873
step  150 |   102s | loss 2.179
step  175 |   120s | loss 2.456
step  200 |   136s | loss 1.881


## 4. Before vs after, on the same template

In [ ]:
for q in questions:
    print(f"PROMPT    : {q}")
    print(f"BASE      : {complete(base, prompt_text(q))!r}")
    print(f"OUR SFT   : {complete(model, prompt_text(q))!r}")
    print(f"INSTRUCT  : {chat(instruct, q)!r}\n")

Try your own question, including some that are *not* in the style of Dolly:

In [ ]:
q = "Give me two tips for learning to cook."
print("OUR SFT :", complete(model, prompt_text(q), max_new_tokens=100))
print("INSTRUCT:", chat(instruct, q, max_new_tokens=100))